## 1. Definitions


In [1]:
from pathlib import Path
import os
import shutil
import json
import re
import pandas as pd
import subprocess
import plotly as pl
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import altair as alt
import ast

#---Data---#
#---Data---#
BMS = ["nodeapp","mediawiki","compression","dacapo-spring","dacapo-luindex","dacapo-lusearch","renaissance-http","renaissance-chirper",
       "502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000","505.mcf_r.inp","523.xalancbmk_r.xalanc","531.deepsjeng_r.ref","541.leela_r.ref"]

CONF = [2,4,6,8,12,16,24,32,40,50,60,75,100,125,150,175,200]

STATS_PREFIX = {
    "multicore": "board.processor.cores1.core.",
    "singlecore": "board.processor.cores.core."
}

PDFS = {
    "penalty_lin": "valuePred.penaltyCycles::",
    "savings_lin": "valuePred.valuePredSavedCycles::",
    "penalty_log": "valuePred.penaltyCyclesLog2::",
    "savings_log": "valuePred.valuePredSavedCyclesLog2::",
}

STATS = {
    "ipc": "ipc",
    "coverage": "valuePred.predCoverage",
    "accuracy": "valuePred.accuracy",
    "penalty_total": "valuePred.totalPenaltyCycles",
    "savings_total": "valuePred.totalSavedCycles",
    "pdfs": PDFS
}

data = {
    "script_file": "../scripts/run_important_simpoints.sh",
    "config_file": "../gem5-configs/all-simpoint-run.py",
    "configs_path": "../gem5-configs/conf-configs",
    "stats_file" : "stats.txt",
    "min-confidence" : 2,
    "max-confidence" : 31,
    "confidence" : CONF, 
    "bms" : BMS,
    "experiment" : "noStride_newConf_delay8_rtz",
    "stats" : STATS,
    "stats_prefix" : STATS_PREFIX,
    "use_stride" : False
}

data["result_path"]= f"results/{data["experiment"]}/"
data["stats_path"] = f"../scripts/results/arm64/{data["experiment"]}/"

with open("config.json", "w") as f:
    json.dump(data, f, indent=4)

pio.templates.default = "simple_white"


#---CHECKS---#
if data["min-confidence"]>=data["max-confidence"]:
    raise ValueError("Check Confidence values")


#---METHODS----#

def contains_line(path,target_line):
    lines = Path(path).read_text().splitlines()
    if target_line in lines:
        return True
    else:
        return False
    
def contains_regex(path,target_regex):
    lines = Path(path).read_text().splitlines()
    return any(re.search(target_regex,line) for line in lines)

def replace_line(path,target_string,new_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_string):
            lines[i] = new_line
            updated = True
            break
    if not updated:
        raise ValueError(f"Key '{target_string}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")


def insert_line_after(path,target_line,new_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_line):
            lines.insert(i+1,new_line)
            updated = True
            break
    if not updated:
        raise ValueError(f"Key '{target_line}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")

def remove_line(path,target_line):
    lines = Path(path).read_text().splitlines()
    updated = False
    for i, line in enumerate(lines):
        if line.strip().startswith(target_line):
            lines.remove(line)
            updated = True
            break
    if not updated:
        print(f"Key '{target_line}' not found in {path}")
    Path(path).write_text("\n".join(lines) + "\n")



## 2. Plotting


### 2.1 Speedup

In [8]:
pio.templates.default = "simple_white"

x_col = "confidence"
y_col = "ipc"
name = "base"
base = pd.read_csv(f"results/baseCase_delay8/results.csv")
experiments= ["noStride_newConf_delay8_rtz_64KiBTab"]#["noStride_newConf_delay8","noStride_newConf_delay8_rtz","noStride_newConf_delay8_64KiBTab","noStride_newConf_delay8_rtz_64KiBTab"]
for exp in experiments:
    df=pd.DataFrame()
    for file in os.listdir(Path(data["result_path"][:-len(data["experiment"])-1]+exp+"/")):
        if not "base" in file:
            file_path = os.path.join(Path(data["result_path"][:-len(data["experiment"])-1]+exp+"/",file))
            temp=pd.read_csv(file_path)
            name_bench=file[:-4][8:]
            temp["Benchmark"]= name_bench
            base_ipc = base.loc[base["benchmark"]==name_bench,y_col].iloc[0]
            temp["speedup"]= temp[y_col] / base_ipc
            df = pd.concat([df,temp], ignore_index=True)

            df.reset_index()
            df.to_csv("temp.csv",index=False)
    #df=df[df["Benchmark"]=="renaissance-chirper"]
    fig=go.Figure()
    fig = px.line(
        df,
        x=x_col,
        y="speedup",
        color="Benchmark",
        markers=True
    )
    fig.update_layout(
        title={
        "text": f"Speedup compared to Baseline for different benchmarks",
        "x": 0.5,
        "font": {"size": 24}
        },
        xaxis_title="Confidence Threshold",
        yaxis_title="Speedup"
    )
    fig.update_xaxes(showgrid=True, gridcolor="lightgray", gridwidth=1,title_font=dict(size=20))
    fig.update_yaxes(showgrid=True, gridcolor="lightgray", gridwidth=1,title_font=dict(size=20))
    fig.show()
    fig.write_image(f"temp/plot_{exp}.png", width=1600, height=800, scale=2)
#----#

base = pd.read_csv(f"results/baseCase_delay3/results.csv")
experiments=[]# ["noStride_newConf_delay3","noStride_newConf_delay3_rtz","noStride_newConf_delay3_64KiBTab","noStride_newConf_delay3_rtz_64KiBTab"]
for exp in experiments:
    df=pd.DataFrame()
    for file in os.listdir(Path(data["result_path"][:-len(data["experiment"])-1]+exp+"/")):
        if not "base" in file:
            file_path = os.path.join(Path(data["result_path"][:-len(data["experiment"])-1]+exp+"/",file))
            temp=pd.read_csv(file_path)
            name_bench=file[:-4][8:]
            temp["Benchmark"]= name_bench
            base_ipc = base.loc[base["benchmark"]==name_bench,y_col].iloc[0]
            temp["speedup"]= temp[y_col] / base_ipc
            df = pd.concat([df,temp], ignore_index=True)

            df.reset_index()
            df.to_csv("temp.csv",index=False)
    fig=go.Figure()
    fig = px.line(
        df,
        x=x_col,
        y="speedup",
        color="Benchmark",
        markers=True
    )
    fig.update_layout(
        title={
        "text": f"Speedup compared to Baseline for multiple Benchmarks",
        "x": 0.5
        },
        xaxis_title="Confidence",
        yaxis_title="Speedup"
    )
    fig.update_xaxes(showgrid=True, gridcolor="lightgray", gridwidth=1,title_font=dict(size=50))
    fig.update_yaxes(showgrid=True, gridcolor="lightgray", gridwidth=1,title_font=dict(size=50))
    fig.show()
    fig.write_image(f"temp/plot_{exp}.png", width=4000, height=4000, scale=5)

### 2.2 Val comparisons
#### 2.2.1 Lines

In [2]:
pio.templates.default = "simple_white"

x_col = "confidence"
y_cols = list(data["stats"].keys())
df=pd.DataFrame()

for file in os.listdir(Path(data["result_path"])):
    file_path = os.path.join(Path(data["result_path"],file))
    temp=pd.read_csv(file_path)
    #print(file)
    temp["Benchmark"]= file[:-4]
    df = pd.concat([df,temp], ignore_index=True)

df.reset_index()
#display(df)
fig=go.Figure()

for col in y_cols:
    if col != "pdfs":
        fig = px.line(
            df,
            x=x_col,
            y=col,
            color="Benchmark",
            markers=True
        )
        fig.update_xaxes(showgrid=True, gridcolor="lightgray", gridwidth=1)
        fig.update_yaxes(showgrid=True, gridcolor="lightgray", gridwidth=1)
        fig.show()


#### 2.2.2 Barchart

In [10]:
pio.templates.default = "simple_white"

y_cols = {"speedup","coverage","accuracy","penalty_total","savings_total"}
confidence=50
base = pd.read_csv(f"results/baseCase_delay8/results.csv")

df=pd.DataFrame()
fig=go.Figure()

for file in os.listdir(data["result_path"]):
    if not "base" in file:
        file_path = os.path.join(data["result_path"],file)
        temp=pd.read_csv(file_path)
        print(file)
        name_bench=file[:-4][8:]
        temp["benchmark"]= name_bench
        base_ipc = base.loc[base["benchmark"]==name_bench,"ipc"].iloc[0]
        temp["ipc"]= temp["ipc"] / base_ipc
        df = pd.concat([df,temp], ignore_index=True)



df.reset_index()
df=df[df["confidence"]==confidence]

### Transform table
df=df.rename(columns={"ipc":"speedup"})
df["penalty_total"]=df["penalty_total"]/df["penalty_total"].max()
df["savings_total"]=df["savings_total"]/df["savings_total"].max()



for col in y_cols:
    fig.add_bar(
        x=df["benchmark"],
        y=df[col],
        name=col
    )


fig.update_layout(
    barmode="group", 
    title=f"Benchmark Comparison with confidence {confidence}",
    xaxis_title="Benchmark",
    yaxis_title="Values"
)


fig.update_xaxes(showgrid=True, gridcolor="lightgray", gridwidth=1)
fig.update_yaxes(showgrid=True, gridcolor="lightgray", gridwidth=1)

fig.show()


results_compression.csv
results_nodeapp.csv
results_505.mcf_r.inp.csv
results_dacapo-spring.csv
results_523.xalancbmk_r.xalanc.csv
results_renaissance-chirper.csv
results_502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000.csv
results_dacapo-lusearch.csv
results_531.deepsjeng_r.ref.csv
results_541.leela_r.ref.csv
results_mediawiki.csv
results_dacapo-luindex.csv
results_renaissance-http.csv



### 2.3 CDFs for Savings and Penalty


In [13]:
pio.templates.default = "simple_white"
#---Set-Desired-Confidence---#
confidence=50
dfs_lin=[]
dfs_log=[]

for file in os.listdir(data["result_path"]):
    if not "base" in file:
        file_path = os.path.join(data["result_path"],file)
        temp=pd.read_csv(file_path)
        temp=temp[temp["confidence"]==confidence]
        
        dfs={"log":[],"lin":[]}
        for col in data["stats"]["pdfs"]:
            temp[col]=temp[col].apply(ast.literal_eval)
            temp_l = temp[col].apply(pd.Series).stack().to_frame(col)
            temp_l[col+"_csum"]=temp_l[col].cumsum()/temp_l.iloc[-1].iloc[0]
            if "lin" in col:
                dfs["lin"].append(temp_l)
            elif "log" in col:
                dfs["log"].append(temp_l)
        
        for df in dfs:
            temp=pd.concat(dfs[df],axis=1).reset_index(level=1).rename(columns={"level_1": "cycles"})
            temp=temp[temp["cycles"]!="total"]
            temp["Benchmark"]= file[:-4][8:]
            if df == "lin":
                dfs_lin.append(temp)
            elif df == "log":
                dfs_log.append(temp)

df_lin=pd.concat(dfs_lin)
df_log=pd.concat(dfs_log)

df_lin.to_csv("temp.csv",index=False)

for pdf in data["stats"]["pdfs"]:
    if "lin" in pdf:
        log=False
        df=df_lin
    elif "log" in pdf:
        log=True
        df=df_log
    fig = px.line(
        df,
        x="cycles",
        y=pdf+"_csum",
        color="Benchmark",
        markers=True,
        log_x=log
    )
    fig.update_xaxes(showgrid=True, gridcolor="lightgray", gridwidth=1, title="Saved Cycles")
    fig.update_yaxes(showgrid=True, gridcolor="lightgray", gridwidth=1, title="Percentage")
    fig.update_layout(
        title={
        "text": f"Cumulative Saved Cycles",
        "x": 0.5,
        "font": {"size": 24}
        },
        xaxis_title_font={"size": 20},
        yaxis_title_font={"size": 20}
    )
    fig.show()
    fig.write_image(f"temp/plot_{pdf}.png", width=1600, height=1000, scale=2)

### 2.4 Comparison of different Experiments

In [8]:
pio.templates.default = "simple_white"

exps = ["base_d8","noStride_newConf_delay8","noStride_newConf_delay8_64KiBTab","noStride_newConf_delay8_rtz","noStride_newConf_delay8_rtz_64KiBTab"]
confidence=50
base = pd.read_csv(f"results/baseCase_delay8/results.csv")
name_base="base_d8"
base["experiment"]=name_base
values=[stat for stat in data["stats"] if stat != "pdfs"]
calc_speedup=True and ("ipc" in values)
if calc_speedup:
    values.append("speedup")

#---Decide if including base (for ipc)---#
df=pd.DataFrame()

for exp in exps:
    if exp == name_base:
        continue
    temp2=pd.DataFrame()
    for file in os.listdir(f"results/{exp}/"):
        if not "base" in file:
            file_path = os.path.join(f"results/{exp}/",file)
            temp=pd.read_csv(file_path)
            #print(file)
            name_bench=file[:-4][8:]
            
            if calc_speedup:
                temp["speedup"]=temp["ipc"]/base["ipc"][base["benchmark"]==name_bench].iloc[0]
            temp=temp[values][temp["confidence"]==confidence]
            temp["benchmark"]= name_bench
            temp2 = pd.concat([temp2,temp], ignore_index=True)
    temp2["experiment"]=exp
    df = pd.concat([df,temp2], ignore_index=True)


df.reset_index()
df.to_csv("temp/df.csv")

for value in values:
    fig=go.Figure()
    for exp in exps:
        fig.add_bar(
            x=df["benchmark"],
            y=df[value][df["experiment"]==exp],
            name=exp
        )

    fig.update_layout(
        barmode="group", 
        title=f"Experiment Comparison with confidence {confidence} and Value {value}",
        xaxis_title="Benchmark",
        yaxis_title=value
    )

    fig.update_xaxes(showgrid=True, gridcolor="lightgray", gridwidth=1)
    fig.update_yaxes(showgrid=True, gridcolor="lightgray", gridwidth=1)

    fig.show()




### 2.5 Load Data Visualization
#### Total

In [11]:
import plotly as pl
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import pandas as pd
pio.templates.default = "simple_white"

def split(df):
    mask = df["pc"] == "-1"
    groups = mask.cumsum()
    return [g for _, g in df[~mask].groupby(groups)]

###Specifiy Parameters###
exp="loadStats_conf50_rtz_3"
#bm="nodeapp"
bm="502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000"
bm="523.xalancbmk_r.xalanc"
name_file="load_stats.csv"

path=data["stats_path"][:-len(data["experiment"])-1]+exp+"/"+bm+"/"+name_file
print(path)

df = pd.read_csv(path,dtype={"pc": str})
dfs = split(df)
df = dfs[1]

df["incoex"]=df["incorrect"]/df["exec"]
df["incopred"]=df["incorrect"]/df["pred"]
df["ps-ratio"]=df["penalty"]/df["savings"]
df["ps-diff"]=df["penalty"]-df["savings"]
df["avg-penalty"]=df["penalty"]/df["incorrect"]
df["avg-savings"]=df["savings"]/df["correct"]


df.to_csv("temp/temp.csv")

num=15
top = df.sort_values("savings", ascending=False).head(num)
#top["pc_s"] = top["pc"].astype(str)

fig = go.Figure()
fig.add_bar(x=top["pc"], y=top["exec"], name="exec", marker_color="darkslategray")
fig.add_bar(x=top["pc"], y=top["pred"], name="pred", marker_color="navy")
fig.add_bar(x=top["pc"], y=top["penalty"], name="penalty", marker_color="tomato")
fig.add_bar(x=top["pc"], y=top["savings"], name="savings", marker_color="yellowgreen")
fig.add_bar(x=top["pc"], y=top["incorrect"], name="incorrect", marker_color="firebrick")
fig.add_bar(x=top["pc"], y=top["correct"], name="correct", marker_color="green")

fig.update_layout(
    barmode="group",
    title=f"Top {num} Loads from {bm}, {exp}",
    xaxis_title="PC of Loads",
    yaxis=dict(
        title="Values",
        side="left",
        type="log",   
        showgrid=True
    ),   
)
fig.show()
display(top)


../scripts/results/arm64/loadStats_conf50_rtz_3/523.xalancbmk_r.xalanc/load_stats.csv


,pc,exec,pred,correct,incorrect,penalty,savings,incoex,incopred,ps-ratio,ps-diff,avg-penalty,avg-savings
8615,7656632,350378,350378,350378,0,0,63607990,0.000000,0.000000,0.000000,-63607990,NaN,181.541050
6889,7656240,30313,30313,30313,0,0,6224755,0.000000,0.000000,0.000000,-6224755,NaN,205.349355
4638,7656236,30313,28100,28068,32,522,5907857,0.001056,0.001139,0.000088,-5907335,16.312500,210.483718
4634,7657172,341698,341295,341286,9,89,2997152,0.000026,0.000026,0.000030,-2997063,9.888889,8.781937
7934,6944628,50069,50069,50069,0,0,1476772,0.000000,0.000000,0.000000,-1476772,NaN,29.494737
8305,5266196,12346,10602,10576,26,351,1229522,0.002106,0.002452,0.000285,-1229171,13.500000,116.255862
6511,8001440,51652,51652,51652,0,0,1161017,0.000000,0.000000,0.000000,-1161017,NaN,22.477678
7738,8001476,50079,9430,9422,8,677,1026306,0.000160,0.000848,0.000660,-1025629,84.625000,108.926555
7741,8001460,233677,9454,9446,8,88,1020200,0.000034,0.000846,0.000086,-1020112,11.000000,108.003388
5963,6961684,50069,9430,9422,8,35,991387,0.000160,0.000848,0.000035,-991352,4.375000,105.220442


#### Ratios

In [14]:
import plotly as pl
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import pandas as pd
pio.templates.default = "simple_white"

def split(df):
    mask = df["pc"] == "-1"
    groups = mask.cumsum()
    return [g for _, g in df[~mask].groupby(groups)]

###Specifiy Parameters###
exp="loadStats_conf50_rtz_3"
bm="nodeapp"
#bm="502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000"
name_file="load_stats.csv"

path=data["stats_path"][:-len(data["experiment"])-1]+exp+"/"+bm+"/"+name_file
print(path)

df = pd.read_csv(path,dtype={"pc":str})
dfs = split(df)
df = dfs[1]
df.to_csv("temp/temp.csv")

df["incoex"]=df["incorrect"]/df["exec"]
df["incopred"]=df["incorrect"]/df["pred"]
df["ps-ratio"]=df["penalty"]/df["savings"]
df["ps-diff"]=df["penalty"]-df["savings"]
df["avg-penalty"]=df["penalty"]/df["incorrect"]

top = df.sort_values("ps-ratio", ascending=False).head(20)
#top = top.sort_values("penalty", ascending=False).head(20)


fig = go.Figure()
#fig.add_bar(x=top["pc"], y=top["exec"], name="exec", marker_color="darkslategray")
fig.add_bar(x=top["pc"], y=top["pred"]/top["exec"], name="pred/exec", marker_color="navy")
#fig.add_bar(x=top["pc"], y=top["penalty"], name="penalty", marker_color="tomato")
#fig.add_bar(x=top["pc"], y=top["ps-ratio"], name="ps-ratio", marker_color="yellowgreen")
fig.add_bar(x=top["pc"], y=top["incoex"], name="incorrect/exec", marker_color="firebrick")
fig.add_bar(x=top["pc"], y=top["correct"]/top["exec"], name="correct/exec", marker_color="green")


fig.update_layout(
    barmode="group",
    title=f"Top Loads from {bm}, {exp}",
    xaxis_title="PC of Loads",
    yaxis=dict(
        title="Values",
        side="left",
        #type="log",   
        showgrid=True
    ),
    
)
fig.show()
display(top)


../scripts/results/arm64/loadStats_conf50_rtz_3/nodeapp/load_stats.csv


,pc,exec,pred,correct,incorrect,penalty,savings,incoex,incopred,ps-ratio,ps-diff,avg-penalty
116714,187650015994264,1431,1,0,1,18,0,0.000699,1.000000,inf,18,18.000
116666,187650015910028,278,1,0,1,18,0,0.003597,1.000000,inf,18,18.000
218210,187650015994224,1062,1,0,1,19,0,0.000942,1.000000,inf,19,19.000
139734,187650015058576,1169,1,0,1,20,0,0.000855,1.000000,inf,20,20.000
139125,187650014943780,401,2,0,2,36,0,0.004988,1.000000,inf,36,18.000
116602,187650015914580,300,1,0,1,15,0,0.003333,1.000000,inf,15,15.000
141095,187650014536800,5118,2,0,2,27,0,0.000391,1.000000,inf,27,13.500
136966,281474839824956,1110,1,0,1,19,0,0.000901,1.000000,inf,19,19.000
136941,187650016505512,7367,1,0,1,12,0,0.000136,1.000000,inf,12,12.000
131803,281474839824064,22954,1,0,1,19,0,0.000044,1.000000,inf,19,19.000


## 4. Complete Cleanup
#### 4.1 CSV Results

In [3]:
path = Path(data["result_path"])
if path.exists():
    shutil.rmtree(path)

#### 4.2 Simulation Results (USE CAREFULLY)

In [12]:
#shutil
path = Path(data["stats_path"])
if path.exists():
    confirm = input("Confirm deleting of simulation Results? (yes/no): ")
    if confirm =="yes":
        shutil.rmtree(path)
        print("Results deleted")
    else:
        print("Cancelled")

Cancelled
